# Agent 智能体设计模式

## 评估器-优化器(Evaluator-optimizer)
在评估器-优化器工作流中，一个 LLM 调用生成响应，而另一个调用在循环中提供评估和反馈。

![](https://i-blog.csdnimg.cn/direct/39f8f760d12346d99ae43fcb9913fbe8.png)


In [2]:
from openai import OpenAI
from datetime import datetime
import json
from typing import List, Dict, Callable
import os
import re
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
load_dotenv("/Users/a1-6/Documents/projects/DL/.env")
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

llm_name = llm_name = "qwen-plus"
def call_llm(user_prompt, system_prompt=""):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user","content": user_prompt}
        ]
    response = client.chat.completions.create(
            model= llm_name,
            messages = messages)
    return(response.choices[0].message.content)

In [1]:
def extract_xml(text: str, tag: str) -> str:
    match = re.search(f'<{tag}>(.*?)</{tag}>', text, re.DOTALL)
    return match.group(1) if match else ""


In [3]:
generator_prompt="""
任务：
根据提供的主题写一篇论证文章。请遵循以下结构和步骤，确保文章逻辑严密、有说服力。
1. 引言部分
开头简要介绍背景信息，说明该议题的重要性。清晰明确地提出文章的中心论点，即你要支持或反对的观点。
2. 主体部分
主体部分应分为多个段落，每个段落围绕一个理由展开，支撑你的中心论点。
3. 结论部分
总结你的中心论点和主要支持理由，提醒读者文章的关键观点。
4. 注意事项
保持语言简洁明了，避免过于复杂或模糊的表达。
确保论证严谨，避免情感化的语言或空洞的说法。
使用可信的来源来支持论点，避免无根据的主张。
字数要求 1000 字左右

如果有任何的反馈，你需要基于反馈来修改。

你需要使用下面的XML标签来输出两部分内容。

<thoughts>
这里写你对主题的理解，对反馈内容的理解，以及如何来修改内容。
</thoughts>

<response>I
论文内容写在这里
</response>
"""


In [4]:
evaluator_prompt="""
请根据以下标准，评估输入的论证文章的质量和逻辑性：

1. 明确性与中心论点：
论证的中心论点是否清晰明确？是否能够轻松识别并理解论点？
论证是否围绕一个可辩论的主题展开，避免陈述无争议的事实？
2. 准确性与前提：
文章中的前提是否真实且可靠？前提是否有明确的证据支持？
是否存在任何事实性错误或不准确的描述？
3. 推理与逻辑性：
论证的推理是否合理？是否能从前提出发，合逻辑地推导出结论？
是否存在逻辑谬误，例如因果谬误、以偏概全等？
4. 证据支持与相关性：
论证中的证据是否充分且相关？每个支持理由是否有可靠的证据（例如专家意见、数据、案例）支持？
是否有证据或反例被遗漏？证据是否直接支持论点？
5. 一致性与逻辑结构：
论证是否符合以下批判性思维标准：
清唽性：表达是否简洁明了？
精确性：语言是否具体准确，避免模糊表述？
相关性：所有内容是否与论点紧密相关？
一致性：论证内部是否一致，没有相互矛盾的主张？
完整性：论证是否考虑了所有相关证据？
公平性：辩论者是否公平处理反对意见，是否无偏见？
6. 反驳策略与对反对意见的回应：
论证是否回应了反对意见？如果有，回应是否充分且合逻辑？
论证是否忽略了相反证据，或者存在关键的反对意见没有被讨论？
7. 结论的支持：
论证的结论是否得到前提和证据的充分支持？结论是否合理、可靠？

根据上述标准，总体评价该论证文章的质量，给出两部分反馈内容。
根据文章的论证质量输出打分，打分是 0 到 100 的区间
指出论证中的弱点，并给出改进的建议。

你需要使用下面的XML标签格式来输出评分和反馈。

<evaluation>
这里输出论文评分
</evaluation>

<feedback>
这里输出反馈建议
</feedback>
"""


In [5]:
task="AI的迅速发展会帶来大量失业吗"

In [6]:
def generate(prompt: str, task: str, context: str = "") -> tuple[str, str]:
    """Generate and improve a solution based on feedback."""
    full_prompt = f"{prompt}\n{context}\n文章主题如下:{task}" if context else f"{prompt}\n{task}"
    response = call_llm(full_prompt)
    thoughts = extract_xml(response, "thoughts")
    result = extract_xml(response, "response")
    print("\n=== GENERATION START ===")
    print(f"Thoughts:\n{thoughts}\n")
    print(f"Generated:\n{result}")
    print("=== GENERATION END ===\n")
    return thoughts, result

In [7]:
def evaluate(prompt: str, content: str, task: str) -> tuple[str, str]:
    """Evaluate if a solution meets requirements."""
    full_prompt = f"{prompt}\n文章主题如下:{task}\n要评估的内容如下:{content}"
    response = call_llm(full_prompt)
    evaluation = extract_xml(response, "evaluation")
    feedback = extract_xml(response, "feedback")
    print("=== EVALUATION START ===")
    print(f"Score: {evaluation}")
    print(f"Feedback: {feedback}")
    print("=== EVALUATION END ===\n")
    return evaluation, feedback

In [8]:
def loop(task: str, evaluator_prompt: str, generator_prompt: str) -> tuple[str, list[dict]]:
    """Keep generating and evaluating until requirements are met."""
    memory = []
    chain_of_thought = []
    # 生成初稿
    thoughts, result = generate(generator_prompt, task)
    memory.append(result)
    chain_of_thought.append({"thoughts": thoughts, "result": result})
    # 进入迭代
    while True:
        # 评估文章内容
        evaluation, feedback = evaluate(evaluator_prompt, result, task)
        # 评估通过则退出
        if float(evaluation) >= 90:
            return result, chain_of_thought
        # 将之前内容和反馈进行拼接,作为背景知识
        context = "\n".join([
            "之前的写作内容如下:",
            *[f"- {m}" for m in memory],
            f"\n反馈如下:\n{feedback}"
        ])
        # 根据反馈重新生成
        context = "\n".join([
                    "之前的写作内容如下:",
                    *[f"- {m}" for m in memory],
                    f"\n反馈如下: \n{feedback}"
                ])
        # 根据反馈重新生成
        thoughts, result = generate(generator_prompt, task, context)
        memory.append(result)
        chain_of_thought.append({"thoughts": thoughts, "result": result})

In [ ]:
result = loop(task=task, evaluator_prompt=evaluator_prompt, generator_prompt=generator_prompt)

- 辩论场景应用

In [9]:
pos_prompt="""
你是一个辩论场上的选手，你需要根据提供的正方观点进行论证。
保持语言简洁明了，避免过于复杂或模糊的表达。
确保论证严谨，避免情感化的语言或空洞的说法。
使用可信的来源来支持论点，避免无根据的主张。

如果辩论对方有任何的观点意见，你需要回应对方的观点意见并修改自己的论证。你的目标不是赢，而是探寻真理。

你需要使用下面的XML标签来输出。
<response>
正方观点：
正方论证过程：
</response>
"""

In [10]:
neg_prompt = """
你是一个辩论场上的选手，你需要根据提供的反方观点进行论证。
保持语言简洁明了，避免过于复杂或模糊的表达。
确保论证严谨，避免情感化的语言或空洞的说法。
使用可信的来源来支持论点，避免无根据的主张。

如果辩论对方有任何的观点意见，你需要回应对方的观点意见并修改自己的论证。你的目标不是赢，而是探寻真理。

你需要使用下面的XML标签来输出。
<response>
反方观点：
反方论证过程：
</response>
"""

In [11]:
pos_task = "正方观点如下: AI的迅速发展会带来大量失业"
neg_task = "反方观点如下: AI的迅速发展不会带来大量失业"

In [12]:
def pos_generate(prompt: str, task: str, context: str = "") -> str:
    full_prompt = f"{prompt}\n{context}\n{task}" if context else f"{prompt}\n{task}"
    response = call_llm(full_prompt)
    result = extract_xml(response, "response")
    print("\n=== POS START ===")
    print(f"Generated:\n{result}")
    print("=== POS END ===\n")
    return result

In [13]:
def neg_generate(prompt: str, task: str, context: str = "") -> str:
    full_prompt = f"{prompt}\n{context}\n{task}" if context else f"{prompt}\n{task}"
    response = call_llm(full_prompt)
    result = extract_xml(response, "response")
    print("\n=== NEG START ===")
    print(f"Generated:\n{result}")
    print("=== NEG END ===\n")
    return result

In [14]:
def loop(pos_task: str,neg_task:str, pos_prompt: str, neg_prompt: str) -> list:
    memory = []
    # 正方先输出
    result = pos_generate(pos_prompt, pos_task)
    memory.append(result)
    # 进入迭代
    for _ in range(3):
        # 拼接背景内容
        memory_string = "\n".join(memory)
        context = f"双方论证历史信息如下 :\n {memory_string}"
        # 反方输出
        result = neg_generate(neg_prompt, context, neg_task)
        memory.append(result)
        # 拼接背景内容
        memory_string = "\n".join(memory)
        context = f"双方论证历史信息如下 :\n {memory_string}"
        # 正方输出
        result = pos_generate(pos_prompt, context, pos_task)
        memory.append(result)
    return memory

In [ ]:
output = loop(pos_task, neg_task, pos_prompt, neg_prompt)